# Utilizing FAISS Vector Store a Storage for Embeddings.

In [2]:
# Importing the FAISS store and HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\DELL\Downloads\AI_RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#Importing Pickle 

import pickle 

# Loading the playbook chunks 
with open('../data/processed/playbook_chunks.pkl', 'rb') as f:
    playbook_chunks = pickle.load(f)

In [8]:
# Loading the log chunks 
with open('../data/processed/log_chunks.pkl', 'rb') as f:
    log_chunks = pickle.load(f)

In [5]:
# Initializing the embedding model 

embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-V2'
)

embedding_model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5363.36it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
# Creating the vectorstore for playbook_chunks and log_chunks 
playbook_vectorstore = FAISS.from_documents(
    documents = playbook_chunks,
    embedding = embedding_model
    )
print(f" Vector stored created with {playbook_vectorstore.index.ntotal} vectors")

 Vector stored created with 174 vectors


In [11]:
# Calling the vector store 
playbook_vectorstore

In [15]:
playbook_vectorstore.save_local("./faiss_index/playbooks")
print('playbook vector stored in "faiss_index" directory')

playbook vector stored in "faiss_index" directory


### Loading the Vector Store and testing the FAISS Store for retrieval and Similarity Score

In [19]:
load_playbook_store = FAISS.load_local(
    "./faiss_index/playbooks",
    embedding_model,
    allow_dangerous_deserialization = True
)

print(f"loaded playbook vector store contains {load_playbook_store.index.ntotal} vectors")

loaded playbook vector store contains 174 vectors


In [27]:
# simple test for a random query as used before 
query = "How do I contain a ransomware attack?"
results = playbook_vectorstore.similarity_search(query, k=10)
print(results)

[Document(id='db692005-b136-42eb-afd0-953e63b2fe55', metadata={'incident_id': 'IR-2025-0141', 'incident_type': 'Ransomware', 'severity': 'Critical', 'final_status': 'Resolved'}, page_content='Phase Identification: Triage alert, snapshot server\nPhase Containment: Isolate server, block C2\nPhase Eradication: Remove ransomware, secure AD\nPhase Recovery: Restore from backups, monitor for reinfection\nPhase Lessons Learned: Review AD security, train admins'), Document(id='7ad051c8-5217-4c91-bfd1-761cbe9f5b30', metadata={'incident_id': 'IR-2025-0018', 'incident_type': 'Malware Infection', 'severity': 'High', 'final_status': 'Resolved'}, page_content='Phase Identification: Confirm malware via AV, collect IOCs\nPhase Containment: Isolate server, block C2 communication\nPhase Eradication: Remove malware, update AV signatures\nPhase Recovery: Restore files, monitor for reinfection\nPhase Lessons Learned: Review download policies, update detection rules'), Document(id='922a80da-d5d4-430d-b5b6-d

In [28]:
# for Top 10 results 
print(f"Query: {query} \n")
print("Top 10 similar chunks:")
for i, doc in enumerate (results):
    print(f"\n {i + 1}.source: {doc.metadata['incident_id']}")
    print(f"content: {doc.page_content[:200]}...")

Query: How do I contain a ransomware attack? 

Top 10 similar chunks:

 1.source: IR-2025-0141
content: Phase Identification: Triage alert, snapshot server
Phase Containment: Isolate server, block C2
Phase Eradication: Remove ransomware, secure AD
Phase Recovery: Restore from backups, monitor for reinfe...

 2.source: IR-2025-0018
content: Phase Identification: Confirm malware via AV, collect IOCs
Phase Containment: Isolate server, block C2 communication
Phase Eradication: Remove malware, update AV signatures
Phase Recovery: Restore fil...

 3.source: IR-2025-0092
content: Phase Identification: Triage alert, snapshot server
Phase Containment: Isolate server, block C2
Phase Eradication: Remove ransomware, secure Kerberos
Phase Recovery: Restore from backups, monitor for ...

 4.source: IR-2025-0036
content: Phase Identification: Triage alert, snapshot endpoint
Phase Containment: Isolate endpoint, block C2
Phase Eradication: Remove ransomware, patch vulnerabilities
Phase Recovery: Restor

In [29]:
#Similarity search with Score
results_and_score = playbook_vectorstore.similarity_search_with_score(query, k =5)
print("\n Similarity Search and Scores")
for doc, score in results_and_score:
    print(f"\nscore: {score:.3f}")
    print(f"source: {doc.metadata['incident_id']}")
    print(f"content preview: {doc.page_content[:100]}")


 Similarity Search and Scores

score: 1.260
source: IR-2025-0141
content preview: Phase Identification: Triage alert, snapshot server
Phase Containment: Isolate server, block C2
Phas

score: 1.271
source: IR-2025-0018
content preview: Phase Identification: Confirm malware via AV, collect IOCs
Phase Containment: Isolate server, block 

score: 1.276
source: IR-2025-0092
content preview: Phase Identification: Triage alert, snapshot server
Phase Containment: Isolate server, block C2
Phas

score: 1.310
source: IR-2025-0036
content preview: Phase Identification: Triage alert, snapshot endpoint
Phase Containment: Isolate endpoint, block C2


score: 1.310
source: IR-2025-0105
content preview: Phase Identification: Triage alert, snapshot endpoint
Phase Containment: Isolate endpoint, block C2



## Storing Log_Chunks in the FAISS Vector Store 

In [ ]:
log_vectorstore = FAISS.from_documents(
    documents = log_chunks,
    embedding = embedding_model 
)

print(f"Vector store created with {log_vectorstore.index.ntotal} vectors")

In [ ]:
# Example query
query = "How do I contain a ransomware attack?"
results = playbook_vectorstore.similarity_search(query, k=5)
for doc in results:
    print(doc.page_content[:200])
    print("-" * 40)